In [1]:
!git clone https://github.com/binance/binance-public-data.git

Cloning into 'binance-public-data'...
remote: Enumerating objects: 282, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 282 (delta 100), reused 49 (delta 49), pack-reused 153 (from 2)
Receiving objects: 100% (282/282), 6.51 MiB | 19.15 MiB/s, done.
Resolving deltas: 100% (149/149), done.


### Get a list of all coins

In [ ]:
import sys
import csv

sys.path.append('binance-public-data/python')
from utility import get_all_symbols

symbols = get_all_symbols("spot")

usdt_pairs = [s for s in symbols if s.endswith("USDT")]

print(f"Всего USDT пар: {len(usdt_pairs)}")
print(usdt_pairs[:20])

csv_path = "usdt_pairs.csv"

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["symbol"])
    for sym in usdt_pairs:
        writer.writerow([sym])

print(f"CSV сохранён: {csv_path}")


Всего USDT пар: 690
['BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'BCCUSDT', 'NEOUSDT', 'LTCUSDT', 'QTUMUSDT', 'ADAUSDT', 'XRPUSDT', 'EOSUSDT', 'TUSDUSDT', 'IOTAUSDT', 'XLMUSDT', 'ONTUSDT', 'TRXUSDT', 'ETCUSDT', 'ICXUSDT', 'VENUSDT', 'NULSUSDT', 'VETUSDT']
CSV сохранён: usdt_pairs.csv


### Imports + Config

In [3]:
# ==========================================================
# Imports
# ==========================================================

import json
import os
import shutil
import subprocess
from datetime import datetime, UTC
from pathlib import Path

import pandas as pd

from kaggle.api.kaggle_api_extended import KaggleApi


# ==========================================================
# Configuration
# ==========================================================

# Kaggle Dataset
DATASET_ID = "roboeggs/binance-spot-klines-usdt"

# Working directory
WORK_DIR = Path("/kaggle/working")

# Binance downloader
BINANCE_SCRIPT = WORK_DIR / "binance-public-data/python/download-kline.py"

# Coin list
COINS_FILE = WORK_DIR / "usdt_pairs.csv"

# Temporary upload directory
UPLOAD_TMP_DIR = WORK_DIR / "upload"

# Folder where Binance stores downloaded archives
DATA_DIR = UPLOAD_TMP_DIR / "data"

# Status file
STATUS_FILE = UPLOAD_TMP_DIR / "download_status.csv"

# Kaggle metadata
METADATA_FILE = UPLOAD_TMP_DIR / "dataset-metadata.json"

# Download parameters
TIMEFRAME = "1m"

# Upload batch when folder reaches this size
MAX_BATCH_SIZE_GB = 5.0

# ==========================================================
# Automatically determine period
# ==========================================================

today = datetime.now(UTC)

START_DATE = "2017-01-01"

END_DATE = today.strftime("%Y-%m-%d")

# Used inside download_status.csv
PERIOD = START_DATE[:7]

print(f"Period : {PERIOD}")
print(f"Start  : {START_DATE}")
print(f"End    : {END_DATE}")

# ==========================================================
# Prepare folders
# ==========================================================

UPLOAD_TMP_DIR.mkdir(parents=True, exist_ok=True)

Period : 2017-01
Start  : 2017-01-01
End    : 2026-07-13


### KaggleDatasetManager

In [4]:
import zipfile


class KaggleDatasetManager:

    def __init__(self, dataset_id, upload_dir, metadata_file):

        self.dataset_id = dataset_id
        self.upload_dir = Path(upload_dir)
        self.metadata_file = Path(metadata_file)

        self.api = KaggleApi()
        self.api.authenticate()

    # ------------------------------------------------------
    # Create dataset-metadata.json
    # ------------------------------------------------------
    def create_metadata(self):

        metadata = {
            "title": "Binance Spot Klines USDT",
            "id": self.dataset_id,
            "licenses": [
                {
                    "name": "CC0-1.0"
                }
            ]
        }

        with open(self.metadata_file, "w") as f:
            json.dump(metadata, f, indent=4)

    # ------------------------------------------------------
    # Download current dataset
    # ------------------------------------------------------
    def download_dataset(self):

        print("Downloading current dataset...")

        self.api.dataset_download_files(
            self.dataset_id,
            path=str(self.upload_dir),
            unzip=True,
            quiet=False
        )

        print("Dataset downloaded.")



    def download_status(self):
        print("Downloading download_status.csv...")
    
        try:
            self.api.dataset_download_file(
                DATASET_ID,
                "download_status.csv",
                path=str(self.upload_dir)
            )
    
            zip_path = self.upload_dir / "download_status.csv.zip"
    
            if zip_path.exists():
                with zipfile.ZipFile(zip_path, "r") as z:
                    z.extractall(self.upload_dir)
    
                zip_path.unlink()
    
                print("Status restored.")
    
            else:
                print("Status file not found in dataset.")
    
        except Exception as e:
            print(f"Unable to download status file: {e}")
    # ------------------------------------------------------
    # Upload new dataset version
    # ------------------------------------------------------
    def upload_version(self, message):

        print(
            "Uploading folder:",
            self.upload_dir
        )
        
        
        print(
            "Files count:",
            len(
                list(
                    self.upload_dir.rglob("*")
                )
            )
        )

        self.create_metadata()

        print("\nUploading new dataset version...")
        print(message)

        self.api.dataset_create_version(
        
            folder=str(self.upload_dir),
        
            version_notes=message,
        
            dir_mode="zip",
        
            convert_to_csv=False,
        
            delete_old_versions=False,
        
            quiet=False
        )

        print("Upload finished.")

    # ------------------------------------------------------
    # Check whether download_status.csv exists
    # ------------------------------------------------------
    def status_exists(self):

        return STATUS_FILE.exists()

In [5]:
kaggle = KaggleDatasetManager(
    DATASET_ID,
    UPLOAD_TMP_DIR,
    METADATA_FILE
)

kaggle.create_metadata()

In [6]:
kaggle.download_status()

Dataset URL: https://www.kaggle.com/datasets/roboeggs/binance-spot-klines-usdt
Status file not found in dataset.


### StatusManager

In [7]:
class StatusManager:

    def __init__(self, status_file, period):

        self.status_file = Path(status_file)
        self.period = period

        self.df = self.load()


    # ------------------------------------------------------
    # Load status file
    # ------------------------------------------------------
    def load(self):

        if self.status_file.exists():

            print("Loading local status file")

            return pd.read_csv(
                self.status_file
            )


        print("Creating new status file")

        return pd.DataFrame(
            columns=[
                "symbol",
                "period",
                "status"
            ]
        )


    # ------------------------------------------------------
    # Get already downloaded symbols
    # ------------------------------------------------------
    def get_completed_symbols(self):

        if self.df.empty:
            return set()


        completed = self.df[
            (self.df["period"] == self.period)
            &
            (self.df["status"] == "success")
        ]["symbol"]


        return set(
            completed.tolist()
        )


    # ------------------------------------------------------
    # Check symbol
    # ------------------------------------------------------
    def is_completed(self, symbol):

        return symbol in self.get_completed_symbols()


    # ------------------------------------------------------
    # Mark batch as completed
    # ------------------------------------------------------
    def mark_success(self, symbols):

        new_rows = pd.DataFrame(
            {
                "symbol": symbols,
                "period": self.period,
                "status": "success"
            }
        )


        self.df = pd.concat(
            [
                self.df,
                new_rows
            ],
            ignore_index=True
        )


        self.save()


    # ------------------------------------------------------
    # Save status file
    # ------------------------------------------------------
    def save(self):

        self.df.to_csv(
            self.status_file,
            index=False
        )


    # ------------------------------------------------------
    # Statistics
    # ------------------------------------------------------
    def count_completed(self):

        return len(
            self.get_completed_symbols()
        )

### BatchManager

In [8]:
class BatchManager:

    def __init__(
        self,
        data_dir,
        max_size_gb
    ):

        self.data_dir = Path(data_dir)

        self.max_size_gb = max_size_gb

        self.symbols = []


    # ------------------------------------------------------
    # Add downloaded symbol
    # ------------------------------------------------------
    def add(self, symbol):

        self.symbols.append(symbol)


    # ------------------------------------------------------
    # Current batch size
    # ------------------------------------------------------
    def size_gb(self):
    
        total_size = 0
    
    
        print(
            "Scanning:",
            self.data_dir
        )
    
    
        files_count = 0
    
    
        for file in self.data_dir.rglob("*"):
    
            if file.is_file():
    
                files_count += 1
    
                total_size += file.stat().st_size
    
    
        size = total_size / (1024 ** 3)
    
    
        print(
            f"Files: {files_count}"
        )
    
        print(
            f"Size: {size:.3f} GB"
        )
    
    
        return size



    # ------------------------------------------------------
    # Check upload condition
    # ------------------------------------------------------
    def need_upload(self):

        size = self.size_gb()

        print(
            f"Current batch size: {size:.2f} GB"
        )

        return size >= self.max_size_gb



    # ------------------------------------------------------
    # Get current symbols
    # ------------------------------------------------------
    def get_symbols(self):

        return self.symbols.copy()



    # ------------------------------------------------------
    # Clear after upload
    # ------------------------------------------------------
    def clear(self):

        self.symbols = []



    # ------------------------------------------------------
    # Remove downloaded files
    # ------------------------------------------------------
    def cleanup(self):

        if self.data_dir.exists():

            print(
                "Removing temporary data..."
            )

            shutil.rmtree(
                self.data_dir
            )

In [9]:
status = StatusManager(
    STATUS_FILE,
    PERIOD
)

done = status.get_completed_symbols()

print(len(done))

Loading local status file
340


### BinanceDownloader

In [10]:
class BinanceDownloader:

    def __init__(
        self,
        script_path,
        upload_dir,
        timeframe,
        start_date,
        end_date
    ):

        self.script_path = str(script_path)
        self.upload_dir = str(upload_dir)

        self.timeframe = timeframe

        self.start_date = start_date
        self.end_date = end_date



    # ------------------------------------------------------
    # Download one symbol
    # ------------------------------------------------------
    def download(self, symbol):

        print("\n" + "=" * 60)
        print(f"Downloading {symbol}")
        print("=" * 60)


        cmd = [

            "python",
            self.script_path,

            "-t",
            "spot",

            "-s",
            symbol,

            "-i",
            self.timeframe,

            "-startDate",
            self.start_date,

            "-endDate",
            self.end_date,

            "-skip-monthly",
            "0",

            "-skip-daily",
            "1",

            "-folder",
            self.upload_dir
        ]


        try:

            result = subprocess.run(

                cmd,

                input="n\n",

                text=True,

                capture_output=False

            )


            if result.returncode != 0:

                print(
                    f"Download failed: {symbol}"
                )

                return False



            print(
                f"{symbol} downloaded"
            )

            return True



        except Exception as e:

            print(
                f"Error downloading {symbol}: {e}"
            )

            return False

### ETLPipeline

In [11]:
class ETLPipeline:


    def __init__(
        self,
        coins_file,
        status_manager,
        downloader,
        batch_manager,
        kaggle_manager
    ):

        self.coins_file = Path(coins_file)

        self.status = status_manager

        self.downloader = downloader

        self.batch = batch_manager

        self.kaggle = kaggle_manager



    # ------------------------------------------------------
    # Load coins
    # ------------------------------------------------------
    def load_coins(self):

        df = pd.read_csv(
            self.coins_file
        )

        return df["symbol"].tolist()



    # ------------------------------------------------------
    # Upload current batch
    # ------------------------------------------------------
    def upload_batch(self):

        symbols = self.batch.get_symbols()


        if not symbols:

            return



        print("\nUploading batch:")
        print(symbols)



        # Save progress
        self.status.mark_success(
            symbols
        )

        print(
            "UPLOAD FOLDER CONTENT:"
        )
        
        for f in Path(UPLOAD_TMP_DIR).rglob("*"):
            print(f)


        self.kaggle.upload_version(

            message=
            f"{PERIOD} batch: {len(symbols)} symbols"

        )


        # Remove temporary files
        self.batch.cleanup()


        self.batch.clear()



    # ------------------------------------------------------
    # Main pipeline
    # ------------------------------------------------------
    def run(self):


        coins = self.load_coins()


        completed = self.status.get_completed_symbols()


        print(
            f"Already downloaded: {len(completed)}"
        )


        remaining = [

            c for c in coins

            if c not in completed

        ]


        print(
            f"Remaining: {len(remaining)}"
        )



        for symbol in remaining:


            success = self.downloader.download(
                symbol
            )


            if not success:

                continue



            self.batch.add(
                symbol
            )


            if self.batch.need_upload():

                self.upload_batch()



        # --------------------------------------------------
        # Final small batch
        # --------------------------------------------------

        if self.batch.get_symbols():

            print(
                "\nFinal upload..."
            )

            self.upload_batch()



        print(
            "\nPipeline finished"
        )

In [ ]:
## pipeline.run()

# ==========================================================
# Create managers
# ==========================================================


kaggle_manager = KaggleDatasetManager(

    DATASET_ID,

    UPLOAD_TMP_DIR,

    METADATA_FILE

)




if not STATUS_FILE.exists():

    try:

        kaggle_manager.download_dataset()

    except Exception as e:

        print(
            "No previous dataset status found"
        )



status_manager = StatusManager(

    STATUS_FILE,

    PERIOD

)



batch_manager = BatchManager(

    DATA_DIR,

    MAX_BATCH_SIZE_GB

)



downloader = BinanceDownloader(

    BINANCE_SCRIPT,

    UPLOAD_TMP_DIR,

    TIMEFRAME,

    START_DATE,

    END_DATE

)



# ==========================================================
# Run ETL
# ==========================================================


pipeline = ETLPipeline(

    COINS_FILE,

    status_manager,

    downloader,

    batch_manager,

    kaggle_manager

)



pipeline.run()

Loading local status file
Already downloaded: 340
Remaining: 350

Folder already exists! Do you want to overwrite it? y/n  Found 1 symbols
[1/1] - start download monthly BETAUSDT klines 

File not found: https://data.binance.vision/data/spot/monthly/klines/BETAUSDT/1m/BETAUSDT-1m-2017-01.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/BETAUSDT/1m/BETAUSDT-1m-2017-02.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/BETAUSDT/1m/BETAUSDT-1m-2017-03.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/BETAUSDT/1m/BETAUSDT-1m-2017-04.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/BETAUSDT/1m/BETAUSDT-1m-2017-05.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/BETAUSDT/1m/BETAUSDT-1m-2017-06.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/BETAUSDT/1m/BETAUSDT-1m-2017-07.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/B

100%|██████████| 4.99G/4.99G [03:14<00:00, 27.6MB/s]


Upload successful: data.zip (5GB)
Starting upload for file download_status.csv


100%|██████████| 11.2k/11.2k [00:00<00:00, 14.0kB/s]


Upload successful: download_status.csv (11KB)
Upload finished.
Removing temporary data...

Folder already exists! Do you want to overwrite it? y/n  Found 1 symbols
[1/1] - start download monthly TIAUSDT klines 

File not found: https://data.binance.vision/data/spot/monthly/klines/TIAUSDT/1m/TIAUSDT-1m-2017-01.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/TIAUSDT/1m/TIAUSDT-1m-2017-02.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/TIAUSDT/1m/TIAUSDT-1m-2017-03.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/TIAUSDT/1m/TIAUSDT-1m-2017-04.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/TIAUSDT/1m/TIAUSDT-1m-2017-05.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/TIAUSDT/1m/TIAUSDT-1m-2017-06.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/TIAUSDT/1m/TIAUSDT-1m-2017-07.zip

File not found: https://data.binance.vision/data/spot/monthl

...

Folder already exists! Do you want to overwrite it? y/n  Found 1 symbols
[1/1] - start download monthly ZKPUSDT klines 

File not found: https://data.binance.vision/data/spot/monthly/klines/ZKPUSDT/1m/ZKPUSDT-1m-2017-01.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/ZKPUSDT/1m/ZKPUSDT-1m-2017-02.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/ZKPUSDT/1m/ZKPUSDT-1m-2017-03.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/ZKPUSDT/1m/ZKPUSDT-1m-2017-04.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/ZKPUSDT/1m/ZKPUSDT-1m-2017-05.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/ZKPUSDT/1m/ZKPUSDT-1m-2017-06.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/ZKPUSDT/1m/ZKPUSDT-1m-2017-07.zip

File not found: https://data.binance.vision/data/spot/monthly/klines/ZKPUSDT/1m/ZKPUSDT-1m-2017-08.zip

File not found: https://data.binance.vision/dat

100%|██████████| 2.75G/2.75G [01:48<00:00, 27.2MB/s]


Upload successful: data.zip (3GB)
Starting upload for file download_status.csv


100%|██████████| 16.8k/16.8k [00:00<00:00, 20.2kB/s]


Upload successful: download_status.csv (17KB)
Upload finished.
Removing temporary data...

Pipeline finished
